In [0]:
%sql
SELECT * FROM spb.silver_places LIMIT 100;

In [0]:
from pyspark.sql import functions as F

silver = spark.table("workspace.spb.silver_places")

silver.select(
    F.countDistinct("basic_category").alias("n_basic")
).show()

silver.groupBy("basic_category").count().orderBy(F.desc("count")).show(30, truncate=False)

# places per cell: 10th, 50th and 90th percentile
silver.groupBy("h3_cell").count().select(
    F.expr("percentile_approx(count, array(0.1, 0.5, 0.9))").alias("places_per_cell")
).show(truncate=False)

In [0]:
from pyspark.sql import functions as F, Window

def cell_category_counts(df, cat_col="basic_category"):
    """One row per (cell, category) with its place count."""
    return (
        df.withColumn("category", F.coalesce(F.col(cat_col), F.lit("unknown")))
          .groupBy("h3_cell", "category")
          .agg(F.count("*").alias("n"))
    )

def places_per_cell(df):
        return counts.groupBy("h3_cell").agg(
            F.sum("n").alias("n_places"),
            F.count("*").alias("n_categories")
        )

def category_shares(counts):
        w = Window.partitionBy("h3_cell")
        return counts.withColumn("share", F.col("n") / F.sum("n").over(w))
    
def diversity(shares):
        "Shannon entropy in bits: -sum(p * log2 p). abs()"
        return shares.groupBy("h3_cell").agg(
            F.abs(-F.sum(F.col("share") * F.log2("share"))).alias("entropy_bits")
        )
    
def top_categories(counts, n=5):
        w = Window.partitionBy("h3_cell").orderBy(F.desc("n"), F.asc("category"))
        return counts.withColumn("category_rank", F.row_number().over(w)).where(F.col("category_rank") <= n)

In [0]:
silver = spark.table("workspace.spb.silver_places")

counts = cell_category_counts(silver)
shares = category_shares(counts)
stats = places_per_cell(counts).join(diversity(shares), "h3_cell")
top5 = top_categories(counts)

shares.write.format("delta").mode("overwrite").saveAsTable("workspace.spb.gold_cell_category")
stats.write.format("delta").mode("overwrite").saveAsTable("workspace.spb.gold_cell_stats")
top5.write.format("delta").mode("overwrite").saveAsTable("workspace.spb.gold_cell_top5")

# Check against silver
print("silver rows", silver.count())
print("sum of n in category tbl", spark.table("workspace.spb.gold_cell_category").agg(F.sum("n")).first()[0])
print("sum of n_places in stats", spark.table("workspace.spb.gold_cell_stats").agg(F.sum("n_places")).first()[0])
print("cells: silver vs stats", silver.select("h3_cell").distinct().count(), spark.table("workspace.spb.gold_cell_stats").count())

In [0]:
counts.display()